# ColumnTransformer: Applying Multiple Preprocessing Pipelines

## Overview
`ColumnTransformer` allows you to apply different preprocessing pipelines to different columns in parallel. It's essential for handling mixed data types (numerical and categorical) efficiently.

## Key Concepts

### What It Does
- Applies transformations to specific column subsets simultaneously
- Combines results into a single transformed dataset
- Maintains modularity and reproducibility

### Basic Syntax
```python
ColumnTransformer([
    ('name1', transformer1, columns1),
    ('name2', transformer2, columns2),
    ...
])
```

### Parameters
- **transformers**: List of tuples `(name, transformer, columns)`
  - `name`: Identifier for the transformer step
  - `transformer`: Pipeline or transformer object to apply
  - `columns`: List of column names to apply to
- **remainder**: How to handle columns not specified (`'drop'`, `'passthrough'`)
- **sparse_threshold**: Set to `0` for dense output (default returns sparse arrays)

## Workflow Example

1. **Separate columns by type**: Identify numerical vs. categorical columns
2. **Create pipelines**: Build preprocessing pipelines for each type
3. **Combine with ColumnTransformer**: Apply all transformations in parallel
4. **Fit and transform**: Use `fit_transform()` on training data

## Output
- Returns a **NumPy array** (not a DataFrame)
- Feature names are lost; you must reconstruct the DataFrame if needed
- Column order follows the order defined in `ColumnTransformer`

## Advantages
✓ Clean, modular code  
✓ Parallel processing of different column types  
✓ Easy to maintain and extend  
✓ Prevents data leakage (fit on train, transform on test)  

## Important Notes
- `ColumnTransformer` output is **array-based**, not labeled
- To inspect intermediate outputs, fit each transformer separately
- Always fit on training data only, then transform test data

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

num_cols = ['age','income']
cat_cols = ['city']

X = pd.DataFrame({
        'age': [25, 30, 35, 40, 45],
        'income': [50000, 60000, 75000, 80000, 90000],
        'city': ['NYC', 'LA', 'NYC', 'Chicago', 'LA']
})

num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                                                                                 ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                                                 ('onehot', OneHotEncoder(handle_unknown='ignore'))])

ct = ColumnTransformer([
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
], remainder='drop', sparse_threshold=0)

X_prepared = ct.fit_transform(X)

X_prepared

array([[-1.41421356, -1.47029409,  0.        ,  0.        ,  1.        ],
       [-0.70710678, -0.77015405,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.28005602,  0.        ,  0.        ,  1.        ],
       [ 0.70710678,  0.63012604,  1.        ,  0.        ,  0.        ],
       [ 1.41421356,  1.33026608,  0.        ,  1.        ,  0.        ]])


- Common pitfalls
    - ColumnTransformer returns an array — recreate a DataFrame and feature names if you need labeled columns.  
    - If you need to inspect intermediate outputs, fit_transform each transformer separately on the relevant columns.

=================================================================
<br>

# ColumnTransformer and Consolidating the Pipeline

In this section, we will consolidate everything we've done so far into one final script using Scikit-Learn pipelines. This includes:

- Creating a stratified test set
- Handling missing values
- Encoding categorical variables
- Scaling numerical features
- Combining everything using    Pipeline and    ColumnTransformer

This will ensure clean, modular, and reproducible code — perfect for production and education.

-----

Final Preprocessing Code using Scikit-Learn Pipelines

In [5]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer

import pandas as pd
import numpy as np



# 1. Load data
housing = pd.read_csv('california_housing_data.csv')



#------------------------------


# 2. Create a stratified test set based on income category


# Create new column "income categories" -
# Acting as STRATA where specific attribut is "median_income" colm
housing[ 'income_categories'] = pd.cut( housing['median_income'] ,
                                       bins=[ 0 ,2 ,4, 6 , 8, np.inf],
                                        labels=[ 1 ,2 , 3, 4, 5 ] )


# Assume income_cat.. is a column in the dataset created from median_income
shuffle_split = StratifiedShuffleSplit( n_splits=1 , test_size=0.2 , random_state=42)

# This ensures that the income distribution in both sets is similar to that of the full dataset
for test_split_indx , train_split_indx in shuffle_split.split( housing , housing[ 'income_categories'] ):
    test_data = housing.loc[test_split_indx]
    train_data = housing.loc[train_split_indx]
    
    
#removing strata cols
train_data.drop( columns=['income_categories'] , inplace = True)   
test_data.drop( columns=['income_categories'] , inplace = True)  


# Copying the training data to work on
housing = train_data.copy()



#---------------------------------


# 3. Seperating feature and label from train data

housing_label = housing['median_house_value'].copy()

housing_features = housing.drop( columns=['median_house_value'] , axis=1)


#-------------------------------

# 4. Separate numerical and categorical columns names in [ list ]- helps in making 2 pipeines

nums_attributes = housing_features.drop(columns=['ocean_proximity'] , axis=1).columns.to_list()

catg_attribute = ['ocean_proximity']


#-------------------------------

# 5. Pipelines


# Numerical pipeline for nums_attribute
nums_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])


# Catogarical Pipeline for catg_attribute
catg_pipeline = Pipeline([
    # ('encoder' , OriginalEncoder()),  -> if import it
    ('onehot_enc' , OneHotEncoder( handle_unknown='ignore'))   # handle_unknown -> if any new/catd data comes in test data then it (ignore it by making it 0 numerialy)
])

# ------------------------


# 6. Construct Full Pipeline - by ColumnTransformer


# ColumnTransformer -> Applies different preprocessing pipelines to different columns( nums/catogary)

# It applies transformations in parallel to different feature columns

# params: list of tuples (name, transformer, columns)
# - name: identifier for the transformer
# - transformer: pipeline or transformer object to apply
# - columns: list of column names to apply the transformer to

# remainder='passthrough' keeps other columns as-is

# sparse_threshold=0 ensures dense output

full_pipeline = ColumnTransformer([
    ('nums' , nums_pipeline , nums_attributes),    # (name , pipeline , attribte_of that pipeline type)
    ('catg' , catg_pipeline , catg_attribute)
])


# Apply the pipeline of data -> to get final train data but in numpy data
prepared_housing_numpy = full_pipeline.fit_transform(housing_features)




#-----------------------------



# 7. Converting prepared_housing_numpy into dataframe data

housing_prepared = pd.DataFrame(prepared_housing_numpy ,
                                columns=['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 
                                         'households', 'median_income','INLAND','NEAR BAY','NEAR OCEAN','<1H OCEAN'] ,
                                index=housing_features.index)


print( housing_prepared)

       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
3700    0.600263 -0.668257           -1.424234     0.141814        0.367324   
20333   0.285900 -0.649575           -1.583145     3.559411        2.554428   
19723  -1.051392  1.508263            1.356712    -0.868768       -0.932735   
14228   1.258930 -1.373525           -1.185867    -0.569057       -0.606547   
11027   0.864728 -0.850413            0.164878    -0.260021       -0.280359   
...          ...       ...                 ...          ...             ...   
10739   0.819819 -0.957837            0.641612     0.334517       -0.085584   
11300   0.819819 -0.864424           -0.073489     0.449517        1.012661   
6766    0.715031 -0.710293           -0.391311     0.918399        1.540663   
10281   0.869718 -0.808377           -0.629678     0.211969       -0.045691   
4768    0.600263 -0.738317            0.403245     0.283011        0.651272   

       population  households  median_income  INLAN